In [ ]:
import gzip
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import io
import re

In [ ]:
vcf_path = 'data/clinvar_20260208.vcf'
f_name = re.search('clinvar_[0-9]{8}', vcf_path).group()

print(f'file name: {f_name}')

# 일단 다 봅시다

In [ ]:
cnt = 0
with open(vcf_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.startswith('#'): continue
        cols = line.strip().split('\t')
        print(cols)
        cnt += 1
        if cnt >= 5: break

# 뭐야 이거 어떻게 읽어요?
>['1', '66926', '3385321', 'AG', 'A', '.', '.', 'ALLELEID=3544463;CLNDISDB=Human_Phenotype_Ontology:HP:0000547,MONDO:MONDO:0019200,MeSH:D012174,MedGen:C0035334,OMIM:268000,OMIM:PS268000,Orphanet:791;CLNDN=Retinitis_pigmentosa;CLNHGVS=NC_000001.10:g.66927del;CLNREVSTAT=criteria_provided,_single_submitter;CLNSIG=Uncertain_significance;CLNSIGSCV=SCV005419006;CLNVC=Deletion;CLNVCSO=SO:0000159;GENEINFO=OR4F5:79501;MC=SO:0001627|intron_variant;ORIGIN=0']

## 위치 정보
1. 1, 66926: 1번 염색체 66926번째 위치
2. 3385321: Clinvar 아이디
3. 'AG' / 'A': AG가 A로 바뀐 돌연변이 ~~염기 하나 가출함~~
4. 점 두개: 각각 QUAL, FILTER

## 핵심 분석 정보
- ALLELEID 이후 구역입니다.
1. ALLELEID: 대립 유전자 아이디
2. CLNDISDB: 질병 데이터베이스 링크
3. CLNDN: 질병 이름
4. CLNHGVS: HGVS 명명법
5. CLNREVSTAT: 검토 신뢰도
6. CLNSIG: 임상적 유의성
7. CLNVC: 변이 타입
8. GENEINFO: 유전자 정보(이름:Entrez ID)
9. MC: 분자적 영향
10. ORIGIN: 변이의 기원(0: 특정되지 않음)

In [ ]:
# 1. VCF 읽기 (이전의 DtypeWarning 해결 버전)
def read_vcf_full(path):
    with open(path, 'r') as f:
        lines = [l for l in f if not l.startswith('##')]

    df = pd.read_csv(
        io.StringIO(''.join(lines)),
        sep='\t',
        dtype={'#CHROM': str},
        low_memory=False
    ).rename(columns={'#CHROM': 'CHROM'})

    # 2. INFO 컬럼을 딕셔너리로 파싱하는 함수
    def parse_info(info_str):
        # 'KEY=VALUE' 쌍들을 분리하여 딕셔너리 생성
        info_dict = {}
        for item in info_str.split(';'):
            if '=' in item:
                key, value = item.split('=', 1)
                info_dict[key] = value
            else:
                info_dict[item] = True  # 값이 없는 플래그(Flag) 처리
        return info_dict

    # 3. INFO 파싱 적용 및 데이터프레임 확장
    info_df = pd.DataFrame(df['INFO'].apply(parse_info).tolist())

    # 4. 기존 컬럼과 합치기 (INFO 원본은 삭제)
    final_df = pd.concat([df.drop(columns=['INFO']), info_df], axis=1)

    return final_df

# 실행
clinvar_df = read_vcf_full(vcf_path)

In [ ]:
clinvar_df

In [ ]:
clinvar_df.to_csv(f'data/{f_name}.csv', index=False)
print(f'saved: data/{f_name}.csv')